# Trader Performance vs Market Sentiment (Hyperliquid)

This notebook follows the assignment format from the **Data Science Intern project**:
- **Part A**: Data preparation
- **Part B**: Analysis with evidence
- **Part C**: Actionable strategy ideas

Datasets used:
- `fear_greed_index.csv`
- `historical_data.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from decimal import Decimal, InvalidOperation

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 160)

plt.style.use('seaborn-v0_8-whitegrid')

## Part A — Data Preparation

In [ ]:
# A1) Load raw datasets + basic shape
fg_raw = pd.read_csv('fear_greed_index.csv')
hist_raw = pd.read_csv(
    'historical_data.csv',
    dtype={'Trade ID': 'string', 'Timestamp': 'string', 'Order ID': 'string'}
)

print('Fear/Greed raw shape:', fg_raw.shape)
print('Historical raw shape:', hist_raw.shape)

fg_raw.head()

In [ ]:
# A1) Missing values and duplicates report
def quality_report(df, name):
    report = {
        'dataset': name,
        'rows': len(df),
        'columns': df.shape[1],
        'duplicate_rows': int(df.duplicated().sum()),
        'total_missing_cells': int(df.isna().sum().sum()),
    }
    out = pd.DataFrame([report])
    missing_by_col = df.isna().sum().sort_values(ascending=False)
    return out, missing_by_col

fg_report, fg_missing = quality_report(fg_raw, 'fear_greed_index.csv')
hist_report, hist_missing = quality_report(hist_raw, 'historical_data.csv')

print('High-level quality report')
display(pd.concat([fg_report, hist_report], ignore_index=True))

print('
Top missing columns: Fear/Greed')
display(fg_missing.head(10).to_frame('missing_count'))

print('
Top missing columns: Historical')
display(hist_missing.head(10).to_frame('missing_count'))

In [ ]:
# A2) Clean Fear/Greed dataset
fg = fg_raw.rename(columns={
    'timestamp': 'fg_timestamp',
    'value': 'fg_value',
    'classification': 'fg_classification',
    'date': 'fg_date',
})

fg['fg_timestamp'] = pd.to_numeric(fg['fg_timestamp'], errors='coerce').astype('Int64')
fg['fg_value'] = pd.to_numeric(fg['fg_value'], errors='coerce')
fg['fg_classification'] = fg['fg_classification'].astype(str).str.strip().str.title()
fg['fg_date'] = pd.to_datetime(fg['fg_date'], errors='coerce').dt.normalize()

fg_clean = (
    fg.dropna(subset=['fg_date', 'fg_value'])
      .drop_duplicates(subset=['fg_date'], keep='last')
      .sort_values('fg_date')
      .reset_index(drop=True)
)
fg_clean['fg_value'] = fg_clean['fg_value'].astype(int)

print('Fear/Greed cleaned shape:', fg_clean.shape)
fg_clean.head()

In [ ]:
# A2) Clean Historical dataset + timestamp conversion + ID normalization

def sci_to_int_string(value):
    if pd.isna(value):
        return pd.NA
    text = str(value).strip()
    if not text:
        return pd.NA
    try:
        return str(int(Decimal(text)))
    except (InvalidOperation, OverflowError, ValueError):
        return pd.NA

hist = hist_raw.rename(columns={
    'Account': 'account',
    'Coin': 'coin',
    'Execution Price': 'execution_price',
    'Size Tokens': 'size_tokens',
    'Size USD': 'size_usd',
    'Side': 'side',
    'Timestamp IST': 'timestamp_ist',
    'Start Position': 'start_position',
    'Direction': 'direction',
    'Closed PnL': 'closed_pnl',
    'Transaction Hash': 'transaction_hash',
    'Order ID': 'order_id',
    'Crossed': 'crossed',
    'Fee': 'fee',
    'Trade ID': 'trade_id_raw',
    'Timestamp': 'timestamp_raw',
})

for col in ['account', 'coin', 'side', 'direction', 'transaction_hash', 'order_id']:
    hist[col] = hist[col].astype(str).str.strip()

hist['coin'] = hist['coin'].str.upper()
hist['side'] = hist['side'].str.upper()
hist['direction'] = hist['direction'].str.title()

for col in ['execution_price', 'size_tokens', 'size_usd', 'start_position', 'closed_pnl', 'fee']:
    hist[col] = pd.to_numeric(hist[col], errors='coerce')

hist['timestamp_ist'] = pd.to_datetime(hist['timestamp_ist'], format='%d-%m-%Y %H:%M', errors='coerce')
hist['timestamp_utc'] = (
    hist['timestamp_ist']
    .dt.tz_localize('Asia/Kolkata', ambiguous='NaT', nonexistent='NaT')
    .dt.tz_convert('UTC')
)
hist['trade_date'] = hist['timestamp_ist'].dt.normalize()

hist['trade_id'] = hist['trade_id_raw'].map(sci_to_int_string).astype('string')
hist['timestamp_ms_approx'] = hist['timestamp_raw'].map(sci_to_int_string).astype('string')

timestamp_ms_numeric = pd.to_numeric(hist['timestamp_ms_approx'], errors='coerce')
hist['timestamp_approx_utc'] = pd.to_datetime(timestamp_ms_numeric, unit='ms', utc=True, errors='coerce')
hist['timestamp_approx_ist'] = hist['timestamp_approx_utc'].dt.tz_convert('Asia/Kolkata')
hist['timestamp_approx_day_match'] = (
    hist['timestamp_approx_ist'].dt.tz_localize(None).dt.normalize() == hist['trade_date']
)

# Direction helpers
hist['is_long'] = hist['direction'].str.contains('Long', case=False, na=False)
hist['is_short'] = hist['direction'].str.contains('Short', case=False, na=False)
hist['is_win'] = hist['closed_pnl'] > 0
hist['is_loss'] = hist['closed_pnl'] < 0
hist['pnl_nonzero'] = hist['closed_pnl'] != 0

hist_clean = hist.dropna(subset=['timestamp_ist']).reset_index(drop=True)

print('Historical cleaned shape:', hist_clean.shape)
print('Timestamp raw unique values:', hist_clean['timestamp_raw'].nunique(dropna=True))
print('Approx timestamp day match ratio:', round(hist_clean['timestamp_approx_day_match'].mean(), 4))
hist_clean.head()

In [ ]:
# A2) Align datasets by date (daily level)
merged = hist_clean.merge(
    fg_clean[['fg_date', 'fg_value', 'fg_classification']],
    left_on='trade_date',
    right_on='fg_date',
    how='left'
)

coverage = merged['fg_value'].notna().mean()
print('Rows matched with sentiment date:', merged['fg_value'].notna().sum(), '/', len(merged))
print('Coverage ratio:', round(coverage, 4))

# Keep classification normalized for grouping
merged['fg_classification'] = merged['fg_classification'].fillna('Unknown')

merged[['trade_date', 'account', 'coin', 'side', 'closed_pnl', 'fg_classification']].head()

In [ ]:
# A3) Create key metrics
# Sentiment bucket at Fear/Greed level (with Neutral preserved)
def to_sentiment_bucket(x):
    txt = str(x).lower()
    if 'fear' in txt:
        return 'Fear'
    if 'greed' in txt:
        return 'Greed'
    if 'neutral' in txt:
        return 'Neutral'
    return 'Unknown'

merged['sentiment_bucket'] = merged['fg_classification'].map(to_sentiment_bucket)

# Daily per account metrics
account_daily = (
    merged.groupby(['trade_date', 'account', 'sentiment_bucket', 'fg_classification'], as_index=False)
    .agg(
        trades=('account', 'size'),
        total_pnl=('closed_pnl', 'sum'),
        avg_trade_size_usd=('size_usd', 'mean'),
        total_volume_usd=('size_usd', 'sum'),
        wins=('is_win', 'sum'),
        losses=('is_loss', 'sum'),
        pnl_nonzero_count=('pnl_nonzero', 'sum'),
        long_trades=('is_long', 'sum'),
        short_trades=('is_short', 'sum'),
        avg_fee=('fee', 'mean'),
    )
)

account_daily['win_rate'] = np.where(
    account_daily['pnl_nonzero_count'] > 0,
    account_daily['wins'] / account_daily['pnl_nonzero_count'],
    np.nan
)
account_daily['long_short_ratio'] = np.where(
    account_daily['short_trades'] > 0,
    account_daily['long_trades'] / account_daily['short_trades'],
    np.nan
)

# Drawdown proxy from cumulative daily PnL per account
account_daily = account_daily.sort_values(['account', 'trade_date']).reset_index(drop=True)
account_daily['cum_pnl'] = account_daily.groupby('account')['total_pnl'].cumsum()
account_daily['cum_peak'] = account_daily.groupby('account')['cum_pnl'].cummax()
account_daily['drawdown_proxy'] = account_daily['cum_pnl'] - account_daily['cum_peak']

# Market-level daily metrics
daily_market = (
    merged.groupby(['trade_date', 'sentiment_bucket'], as_index=False)
    .agg(
        trades=('account', 'size'),
        unique_accounts=('account', 'nunique'),
        total_volume_usd=('size_usd', 'sum'),
        total_pnl=('closed_pnl', 'sum'),
        avg_trade_size_usd=('size_usd', 'mean'),
        long_ratio=('is_long', 'mean'),
        short_ratio=('is_short', 'mean'),
    )
)

print('account_daily shape:', account_daily.shape)
print('daily_market shape:', daily_market.shape)
account_daily.head()

In [ ]:
# Optional metric check: leverage distribution (if available)
leverage_cols = [c for c in merged.columns if 'lever' in c.lower()]

if leverage_cols:
    print('Leverage-related columns found:', leverage_cols)
    for c in leverage_cols:
        tmp = pd.to_numeric(merged[c], errors='coerce')
        print(f'\n{c} summary:')
        display(tmp.describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95]).to_frame('value'))
else:
    print('No leverage column found in provided historical dataset, so leverage distribution cannot be computed directly.')

## Part B — Analysis (with evidence)

In [ ]:
# B1) Does performance differ between Fear vs Greed days?
perf_by_sentiment = (
    account_daily[account_daily['sentiment_bucket'].isin(['Fear', 'Greed'])]
    .groupby('sentiment_bucket', as_index=False)
    .agg(
        trader_days=('account', 'size'),
        mean_daily_pnl=('total_pnl', 'mean'),
        median_daily_pnl=('total_pnl', 'median'),
        mean_win_rate=('win_rate', 'mean'),
        median_drawdown_proxy=('drawdown_proxy', 'median'),
        mean_trade_count=('trades', 'mean'),
        mean_trade_size=('avg_trade_size_usd', 'mean'),
    )
)

print('Performance comparison: Fear vs Greed')
display(perf_by_sentiment)

In [ ]:
# B1) Charts for performance by sentiment
plot_df = account_daily[account_daily['sentiment_bucket'].isin(['Fear', 'Greed'])].copy()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Mean daily pnl bar
tmp = perf_by_sentiment.set_index('sentiment_bucket')['mean_daily_pnl']
axes[0].bar(tmp.index, tmp.values)
axes[0].set_title('Mean Daily PnL by Sentiment')
axes[0].set_ylabel('PnL')

# Mean win rate bar
tmp2 = perf_by_sentiment.set_index('sentiment_bucket')['mean_win_rate']
axes[1].bar(tmp2.index, tmp2.values)
axes[1].set_title('Mean Win Rate by Sentiment')
axes[1].set_ylabel('Win Rate')

# Distribution of daily pnl (clipped to reduce outlier distortion)
for sentiment, color in [('Fear', '#1f77b4'), ('Greed', '#ff7f0e')]:
    vals = plot_df.loc[plot_df['sentiment_bucket'] == sentiment, 'total_pnl'].clip(-20000, 20000)
    axes[2].hist(vals, bins=60, alpha=0.5, label=sentiment, color=color)
axes[2].set_title('Daily PnL Distribution (clipped)')
axes[2].set_xlabel('PnL')
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# B2) Do traders change behavior based on sentiment?
behavior_by_sentiment = (
    account_daily[account_daily['sentiment_bucket'].isin(['Fear', 'Greed', 'Neutral'])]
    .groupby('sentiment_bucket', as_index=False)
    .agg(
        trader_days=('account', 'size'),
        mean_trades_per_day=('trades', 'mean'),
        mean_trade_size=('avg_trade_size_usd', 'mean'),
        mean_long_short_ratio=('long_short_ratio', 'mean'),
        mean_total_volume=('total_volume_usd', 'mean'),
    )
)

print('Behavior metrics by sentiment')
display(behavior_by_sentiment)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].bar(behavior_by_sentiment['sentiment_bucket'], behavior_by_sentiment['mean_trades_per_day'])
axes[0].set_title('Avg Trades per Trader-Day')

axes[1].bar(behavior_by_sentiment['sentiment_bucket'], behavior_by_sentiment['mean_trade_size'])
axes[1].set_title('Avg Trade Size (USD)')

axes[2].bar(behavior_by_sentiment['sentiment_bucket'], behavior_by_sentiment['mean_long_short_ratio'])
axes[2].set_title('Avg Long/Short Ratio')

plt.tight_layout()
plt.show()

In [ ]:
# B3) Segment traders into 3 groups
trader_features = (
    merged.groupby('account', as_index=False)
    .agg(
        total_trades=('account', 'size'),
        active_days=('trade_date', 'nunique'),
        total_pnl=('closed_pnl', 'sum'),
        avg_trade_size_usd=('size_usd', 'mean'),
        wins=('is_win', 'sum'),
        losses=('is_loss', 'sum'),
        realized_trades=('pnl_nonzero', 'sum'),
    )
)

trader_features['avg_trades_per_day'] = trader_features['total_trades'] / trader_features['active_days'].replace(0, np.nan)
trader_features['win_rate'] = np.where(
    trader_features['realized_trades'] > 0,
    trader_features['wins'] / trader_features['realized_trades'],
    np.nan
)

# Segment 1: frequent vs infrequent
freq_cut = trader_features['avg_trades_per_day'].median()
trader_features['segment_frequency'] = np.where(
    trader_features['avg_trades_per_day'] >= freq_cut,
    'Frequent',
    'Infrequent'
)

# Segment 2: high size vs low size
size_cut = trader_features['avg_trade_size_usd'].median()
trader_features['segment_size'] = np.where(
    trader_features['avg_trade_size_usd'] >= size_cut,
    'HighSize',
    'LowSize'
)

# Segment 3: consistent winners vs inconsistent
trader_features['segment_consistency'] = np.where(
    (trader_features['win_rate'] >= 0.55) & (trader_features['total_pnl'] > 0),
    'ConsistentWinner',
    'Inconsistent'
)

print('Trader feature table:')
display(trader_features.head())

print('Segment counts:')
for col in ['segment_frequency', 'segment_size', 'segment_consistency']:
    print('\n' + col)
    display(trader_features[col].value_counts().to_frame('count'))

In [ ]:
# B3) Segment performance summaries
segment_summaries = {}

for segment_col in ['segment_frequency', 'segment_size', 'segment_consistency']:
    summary = (
        trader_features.groupby(segment_col, as_index=False)
        .agg(
            traders=('account', 'nunique'),
            mean_total_pnl=('total_pnl', 'mean'),
            median_total_pnl=('total_pnl', 'median'),
            mean_win_rate=('win_rate', 'mean'),
            mean_trades_per_day=('avg_trades_per_day', 'mean'),
            mean_trade_size=('avg_trade_size_usd', 'mean'),
        )
    )
    segment_summaries[segment_col] = summary
    print(f'\nSummary for {segment_col}')
    display(summary)

# quick visualization for consistency segment
consistency_plot = segment_summaries['segment_consistency']
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(consistency_plot['segment_consistency'], consistency_plot['mean_total_pnl'])
axes[0].set_title('Mean Total PnL by Consistency Segment')
axes[1].bar(consistency_plot['segment_consistency'], consistency_plot['mean_win_rate'])
axes[1].set_title('Mean Win Rate by Consistency Segment')
plt.tight_layout()
plt.show()

In [ ]:
# B4) At least 3 insights backed by computed evidence
fear_greed_cmp = perf_by_sentiment.set_index('sentiment_bucket')

fear_pnl = fear_greed_cmp.loc['Fear', 'mean_daily_pnl'] if 'Fear' in fear_greed_cmp.index else np.nan
greed_pnl = fear_greed_cmp.loc['Greed', 'mean_daily_pnl'] if 'Greed' in fear_greed_cmp.index else np.nan
fear_win = fear_greed_cmp.loc['Fear', 'mean_win_rate'] if 'Fear' in fear_greed_cmp.index else np.nan
greed_win = fear_greed_cmp.loc['Greed', 'mean_win_rate'] if 'Greed' in fear_greed_cmp.index else np.nan

best_segment = segment_summaries['segment_consistency'].sort_values('mean_total_pnl', ascending=False).head(1)
worst_segment = segment_summaries['segment_consistency'].sort_values('mean_total_pnl', ascending=True).head(1)

print('Insight 1: Sentiment vs performance')
print(f'- Mean daily PnL on Fear days:  {fear_pnl:.2f}')
print(f'- Mean daily PnL on Greed days: {greed_pnl:.2f}')
print(f'- Mean win rate on Fear days:   {fear_win:.3f}')
print(f'- Mean win rate on Greed days:  {greed_win:.3f}')

print('\nInsight 2: Behavior shift by sentiment (from behavior_by_sentiment table)')
display(behavior_by_sentiment)

print('\nInsight 3: Segment-level outcomes')
print('Best consistency segment by mean total PnL:')
display(best_segment)
print('Worst consistency segment by mean total PnL:')
display(worst_segment)

## Part C — Actionable Output

In [ ]:
# C1) Strategy ideas / rules of thumb from findings
strategies = []

if pd.notna(fear_pnl) and pd.notna(greed_pnl):
    if fear_pnl > greed_pnl:
        strategies.append(
            'Rule 1: On Fear days, allow normal trade size/frequency for proven profitable segments; on Greed days, tighten risk limits.'
        )
    else:
        strategies.append(
            'Rule 1: On Greed days, allow normal trade size/frequency for proven profitable segments; on Fear days, tighten risk limits.'
        )

# Use segment outcomes
best_name = best_segment.iloc[0, 0] if len(best_segment) else 'BestSegment'
worst_name = worst_segment.iloc[0, 0] if len(worst_segment) else 'WorstSegment'

strategies.append(
    f'Rule 2: Prioritize accounts in `{best_name}` profile for higher allocation; cap exposure for `{worst_name}` profile until performance stabilizes.'
)

strategies.append(
    'Rule 3: For accounts with low win-rate and high trade frequency, enforce cooldowns (max trades/day) and smaller position sizes.'
)

print('Recommended strategy ideas:')
for i, s in enumerate(strategies, 1):
    print(f'{i}. {s}')

In [ ]:
# C2) Save processed outputs for submission package
fg_clean.to_csv('fear_greed_index_clean.csv', index=False)
hist_clean.to_csv('historical_data_clean.csv', index=False)
account_daily.to_csv('account_daily_metrics.csv', index=False)
daily_market.to_csv('daily_market_metrics.csv', index=False)
trader_features.to_csv('trader_features_segments.csv', index=False)

summary_text = []
summary_text.append('# Trader Behavior Insights Summary')
summary_text.append('')
summary_text.append('## Methodology')
summary_text.append('- Cleaned sentiment and historical trader data.')
summary_text.append('- Aligned by daily date and engineered per-trader daily behavior/performance metrics.')
summary_text.append('- Compared Fear vs Greed behavior and segmented traders into 3 groups.')
summary_text.append('')
summary_text.append('## Key Insights')
summary_text.append(f'- Fear mean daily PnL: {fear_pnl:.2f}; Greed mean daily PnL: {greed_pnl:.2f}.')
summary_text.append(f'- Fear mean win-rate: {fear_win:.3f}; Greed mean win-rate: {greed_win:.3f}.')
summary_text.append('- Segment analysis highlights clear performance differences across consistency profiles.')
summary_text.append('')
summary_text.append('## Strategy Recommendations')
for i, s in enumerate(strategies, 1):
    summary_text.append(f'{i}. {s}')

with open('assignment_summary.md', 'w', encoding='utf-8') as f:
    f.write('
'.join(summary_text))

print('Saved files:')
print('- fear_greed_index_clean.csv')
print('- historical_data_clean.csv')
print('- account_daily_metrics.csv')
print('- daily_market_metrics.csv')
print('- trader_features_segments.csv')
print('- assignment_summary.md')